# SigLIP 2 Zero-Shot + TACS Fusion tr?n SUN397

Notebook ??nh gi? m?t nh?nh zero-shot SigLIP 2 v? th? late-fusion v?i checkpoint TACS Adaptive Top-K ?? train. Notebook kh?ng train l?i TACS, kh?ng fine-tune SigLIP 2 v? kh?ng c?n Internet.

Tr??c khi ch?y tr?n Kaggle, g?n c?c Kaggle Input sau:
- SUN397 dataset v? official partitions nh? notebook TACS g?c.
- DINOv3 code + weights nh? notebook TACS g?c.
- File best_val_checkpoint.pt ?? t?i t? output c?a l?n train TACS adaptive.
- Th? m?c model SigLIP 2 ??y ??: google/siglip2-base-patch16-224, g?m config.json, model.safetensors v? processor/tokenizer files.

S?a hai ???ng d?n TACS_CHECKPOINT_PATH v? SIGLIP2_MODEL_DIR ? cell c?u h?nh. T?n l?p SUN397 ???c d?ng l?m text prompts; ??y l? baseline zero-shot c? d?ng t?n nh?n, kh?c v?i TACS thu?n ?nh.


## 1. C?u h?nh v? SUN397 split

Cell n?y gi? nguy?n loader, seed, preprocessing v? candidate pool 20% t? notebook TACS ?? train. Output ???c l?u sang th? m?c m?i.


In [ ]:
import json
import random
import subprocess
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

torch.backends.cudnn.benchmark = True
# Tận dụng Tensor Cores thế hệ mới trên dòng RTX / GPU Ampere+
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

# --- ĐƯỜNG DẪN OUTPUT (Ghi vào working của Kaggle) ---
PROJECT_DIR = Path('/kaggle/working')
OUT_PUT_DIR = PROJECT_DIR / 'outputs' / 'siglip2_tacs_fusion_sun397'
OUTPUT_DIR = OUT_PUT_DIR
OUT_PUT_DIR.mkdir(parents=True, exist_ok=True)

# --- ĐƯỜNG DẪN DATASET (SUN397) ---
DATASET_ROOT = Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397')
if not DATASET_ROOT.exists():
    fallback_candidates = [
        Path('/kaggle/input/sun397/SUN397/SUN397'),
        Path('/kaggle/input/sun397/SUN397'),
        Path('/kaggle/input/sun397'),
        Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397'),
    ]
    for candidate in fallback_candidates:
        if candidate.exists():
            DATASET_ROOT = candidate
            break

IMAGE_ROOT = DATASET_ROOT

# --- ĐƯỜNG DẪN DINOv3 REPO & WEIGHTS ---
DINOV3_REPO_DIR = '/kaggle/input/datasets/sinhnguync/dinov3-code/dinov3-main'
if not Path(DINOV3_REPO_DIR).exists():
    for p in [Path('/kaggle/input/dinov3-code/dinov3-main'), Path('/kaggle/input/dinov3-code')]:
        if p.exists():
            DINOV3_REPO_DIR = str(p)
            break

DINOV3_MODEL_NAME = 'dinov3_vits16'
DINOV3_WEIGHTS = '/kaggle/input/datasets/sinhnguync/dinov3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'
if not Path(DINOV3_WEIGHTS).exists():
    for p in [
        Path('/kaggle/input/dinov3/dinov3_vits16_pretrain_lvd1689m-08c60483.pth'),
        Path('/kaggle/input/dinov3/dinov3_vits16.pth'),
    ]:
        if p.exists():
            DINOV3_WEIGHTS = str(p)
            break

BACKBONE_NAME = DINOV3_MODEL_NAME
FEATURE_DIM = 384

IMAGE_SIZE = 384
BATCH_SIZE = 64 # K nho ngau nhien 1..3 giup gioi han VRAM
EPOCHS = 100
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
BACKBONE_LR = 3e-6
HEAD_LR = 1e-4
BACKBONE_WEIGHT_DECAY = 0.05
HEAD_WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
TOKEN_HEAD_DROPOUT = 0.3
FREEZE_BACKBONE_EPOCHS = 0
WARMUP_EPOCHS = 5
PATIENCE = 10
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 0
NUM_CLASSES = 397  # SUN397 có 397 scene categories
TOP_K = 5
CONTEXT_TOP_K = 3  # So context toi da
DIVERSITY_WEIGHT = 0.35
ADAPTIVE_SCORE_GAP_THRESHOLD = 0.75
ADAPTIVE_THRESHOLD_GRID = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
RETRIEVAL_SCORE_WEIGHT = 0.0
SUPCON_WEIGHT = 0.05
SUPCON_TEMPERATURE = 0.10
USE_BBOX_CROP = False  # SUN397 là bài toán scene classification, không dùng crop bbox
CANDIDATE_POOL_FRACTION = 0.20
VAL_FRACTION = 0.10
RETRIEVAL_BATCH_SIZE = 512
RETRIEVAL_CHUNK_SIZE = 4096
QUERY_CONTEXT_GATE_DROPOUT = 0.20
QUERY_CONTEXT_GATE_BIAS = -0.7
EVAL_TTA_HORIZONTAL_FLIP = False
USE_CONTEXT_LABEL_PRIOR = False
CONTEXT_LABEL_PRIOR_WEIGHT = 0.25
CONTEXT_LABEL_PRIOR_TEMPERATURE = 0.07

TRAIN_ON_FULL_TRAIN = False
FULL_TRAIN_MONITOR_FRACTION = 0.05  # chi dung khi TRAIN_ON_FULL_TRAIN=True

# --- TACS selector hyperparameters ---
TACS_CANDIDATES_PER_QUERY = None  # None nghĩa là dùng toàn bộ candidate pool
TACS_PREFILTER_TOP_SIMILAR = 0
TACS_GUMBEL_TAU = 0.10
TACS_POLICY_WEIGHT = 1.0
TACS_ENTROPY_WEIGHT = 0.01
TACS_SELECTOR_LR = 1e-4
TACS_SELECTOR_WEIGHT_DECAY = 1e-4
TACS_SELECTOR_DROPOUT = 0.10
TACS_SELECTOR_TEMPERATURE = 1.0

# --- Full-pool selector memory controls ---
TACS_USE_FULL_CANDIDATE_POOL = True
CANDIDATE_IMAGE_BANK_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CANDIDATE_IMAGE_BANK_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
CANDIDATE_FEATURE_BANK_BATCH_SIZE = 128
FULL_POOL_SCORE_CHUNK_SIZE = 1024

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE + 48, IMAGE_SIZE + 48)),
    transforms.RandomResizedCrop(
        (IMAGE_SIZE, IMAGE_SIZE),
        scale=(0.60, 1.0),
        ratio=(0.80, 1.25),
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.25,
            contrast=0.25,
            saturation=0.20,
            hue=0.04,
        )
    ], p=0.75),
    transforms.RandomGrayscale(p=0.05),
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))
    ], p=0.15),
    transforms.ToTensor(),
    transforms.RandomErasing(
        p=0.20,
        scale=(0.02, 0.12),
        ratio=(0.3, 3.3),
        value="random",
    ),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

_mean = globals().get("DINO_MEAN", globals().get("IMAGENET_MEAN", (0.485, 0.456, 0.406)))
_std = globals().get("DINO_STD", globals().get("IMAGENET_STD", (0.229, 0.224, 0.225)))

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=_mean, std=_std),
])


def load_class_names(dataset_root):
    classes_path = dataset_root / 'ClassName.txt'
    if not classes_path.is_file():
        fallback_paths = [
            dataset_root.parent / 'ClassName.txt',
            dataset_root / 'classes.txt',
            Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/SUN397/ClassName.txt'),
            Path('/kaggle/input/sun397/SUN397/SUN397/ClassName.txt'),
        ]
        for p in fallback_paths:
            if p.is_file():
                classes_path = p
                break

    if not classes_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy ClassName.txt tại: {classes_path}")

    class_names = []
    with open(classes_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                # Dòng có dạng /a/abbey hoặc a/abbey -> chuẩn hóa bỏ dấu / ở đầu
                class_names.append(line.lstrip('/'))

    global NUM_CLASSES
    if len(class_names) != NUM_CLASSES:
        print(f"Cập nhật NUM_CLASSES từ {NUM_CLASSES} thành {len(class_names)}")
        NUM_CLASSES = len(class_names)
    return class_names


def load_sun397_annotations(dataset_root, image_root, class_names):
    class_to_id = {cls.lstrip('/'): idx for idx, cls in enumerate(class_names)}

    # 1. Tìm các file partition chuẩn (Training_01.txt / Testing_01.txt) nếu có sẵn trong dataset
    train_part_file, test_part_file = None, None
    search_dirs = [
        dataset_root,
        dataset_root.parent / 'sun397_other',
        dataset_root.parent / 'sun397_other' / 'Partitions',
        dataset_root.parent.parent / 'sun397_other',
        Path('/kaggle/input/datasets/hiuphmnhtrung/sun397/SUN397/sun397_other'),
    ]
    for d in search_dirs:
        if d.is_dir():
            t1 = d / 'Training_01.txt'
            te1 = d / 'Testing_01.txt'
            if t1.is_file() and te1.is_file():
                train_part_file, test_part_file = t1, te1
                break
            found_t = list(d.glob('**/Training_01.txt'))
            if found_t:
                train_part_file = found_t[0]
                cand_te = train_part_file.parent / 'Testing_01.txt'
                if cand_te.is_file():
                    test_part_file = cand_te
                    break

    records = []
    if train_part_file and test_part_file:
        print(f"Sử dụng official partition: {train_part_file.name} và {test_part_file.name}")
        for split_label, p_file in [('train', train_part_file), ('test', test_part_file)]:
            with open(p_file, 'r', encoding='utf-8') as f:
                for line in f:
                    rel_p = line.strip().lstrip('/')
                    if not rel_p:
                        continue
                    class_folder = str(Path(rel_p).parent).replace('\\', '/').lstrip('/')
                    if class_folder not in class_to_id:
                        continue
                    class_id = class_to_id[class_folder]
                    full_p = image_root / rel_p
                    if not full_p.is_file():
                        if (dataset_root / rel_p).is_file():
                            full_p = dataset_root / rel_p
                        else:
                            continue
                    records.append({
                        'rel_path': rel_p,
                        'fname': Path(rel_p).name,
                        'image_path': str(full_p.resolve()),
                        'class_id': int(class_id),
                        'is_train': 1 if split_label == 'train' else 0,
                    })
        df = pd.DataFrame(records)
    else:
        print("Không thấy file partition, tự động quét thư mục ảnh theo ClassName.txt và chia split 50/50 chuẩn...")
        all_rows = []
        rng = np.random.default_rng(SEED)
        valid_exts = {'.jpg', '.jpeg', '.png', '.bmp'}

        for cls_name in tqdm(class_names, desc="Đang đọc các lớp SUN397"):
            cls_dir = image_root / cls_name
            if not cls_dir.is_dir():
                cls_dir = dataset_root / cls_name
            if not cls_dir.is_dir():
                continue

            cls_id = class_to_id[cls_name]
            img_paths = [p for p in cls_dir.iterdir() if p.is_file() and p.suffix.lower() in valid_exts]
            img_paths.sort()
            if not img_paths:
                continue

            n_total = len(img_paths)
            indices = np.arange(n_total)
            rng.shuffle(indices)

            # Benchmark chuẩn SUN397: 50 train / 50 test mỗi class nếu >= 100 ảnh, hoặc chia đôi
            if n_total >= 100:
                n_train = 50
                n_test = 50
                train_idx = set(indices[:n_train])
                test_idx = set(indices[n_train:n_train + n_test])
            else:
                n_train = max(1, n_total // 2)
                train_idx = set(indices[:n_train])
                test_idx = set(indices[n_train:])

            for idx, p in enumerate(img_paths):
                if idx in train_idx:
                    is_train = 1
                elif idx in test_idx:
                    is_train = 0
                else:
                    continue

                try:
                    rel_p = str(p.relative_to(image_root)).replace('\\', '/')
                except Exception:
                    rel_p = f"{cls_name}/{p.name}"

                all_rows.append({
                    'rel_path': rel_p,
                    'fname': p.name,
                    'image_path': str(p.resolve()),
                    'class_id': int(cls_id),
                    'is_train': is_train,
                })
        df = pd.DataFrame(all_rows)

    # Thêm các cột dummy BBox để tương thích 100% với Dataset class ở Cell 3 mà không gây lỗi
    df['image_id'] = np.arange(len(df))
    df['bbox_x1'] = 0
    df['bbox_y1'] = 0
    df['bbox_x2'] = 99999
    df['bbox_y2'] = 99999

    train_df = df[df['is_train'] == 1].copy().reset_index(drop=True)
    train_df['split'] = 'train'
    train_df['row_id'] = np.arange(len(train_df))

    test_df = df[df['is_train'] == 0].copy().reset_index(drop=True)
    test_df['split'] = 'test'
    test_df['row_id'] = np.arange(len(test_df))

    return train_df, test_df


def split_train_val_by_class(df, val_fraction=0.10, seed=42):
    if not 0 < val_fraction < 1:
        raise ValueError('val_fraction phải nằm trong khoảng (0, 1)')
    rng = np.random.default_rng(seed)
    val_indices = []
    for _, group in df.groupby('class_id', sort=True):
        group_indices = group.index.to_numpy()
        if len(group_indices) <= 1:
            continue
        n_val = max(1, int(round(len(group_indices) * val_fraction)))
        n_val = min(n_val, len(group_indices) - 1)
        val_indices.extend(rng.choice(group_indices, size=n_val, replace=False).tolist())
    val_indices = np.array(sorted(val_indices), dtype=int)
    train_split = df.drop(index=val_indices).reset_index(drop=True)
    val_split = df.loc[val_indices].reset_index(drop=True)
    return train_split, val_split


# --- Tải dữ liệu SUN397 ---
class_names = load_class_names(DATASET_ROOT)
full_train_df, test_df = load_sun397_annotations(DATASET_ROOT, IMAGE_ROOT, class_names)
train_df, val_df = split_train_val_by_class(full_train_df, VAL_FRACTION, SEED)

if TRAIN_ON_FULL_TRAIN:
    _, monitor_val_df = split_train_val_by_class(
        full_train_df,
        FULL_TRAIN_MONITOR_FRACTION,
        SEED,
    )
    train_df = full_train_df.reset_index(drop=True)
    val_df = monitor_val_df.reset_index(drop=True)

candidate_rng = np.random.default_rng(SEED)
if CANDIDATE_POOL_FRACTION >= 1.0:
    candidate_pool_positions = np.arange(len(train_df), dtype=int)
else:
    candidate_pool_size = max(1, int(round(len(train_df) * CANDIDATE_POOL_FRACTION)))
    candidate_pool_positions = np.sort(
        candidate_rng.choice(len(train_df), size=candidate_pool_size, replace=False)
    )

candidate_df = train_df.iloc[candidate_pool_positions].reset_index(drop=True)
candidate_df.to_csv(OUT_PUT_DIR / 'candidate_pool.csv', index=False)
val_df.to_csv(OUT_PUT_DIR / 'validation_split.csv', index=False)

print('DATASET_ROOT:', DATASET_ROOT.resolve())
print('IMAGE_ROOT:', IMAGE_ROOT.resolve())
print('OUT_PUT_DIR:', OUT_PUT_DIR.resolve())
print(f'Standard train images: {len(full_train_df):,}')
print(f'Actual train images: {len(train_df):,}')
print(f'Validation images: {len(val_df):,}')
print(f'Train on full train: {TRAIN_ON_FULL_TRAIN}')
print(f'Standard test images: {len(test_df):,}')
print(f'Candidate pool: {len(candidate_df):,}/{len(train_df):,} actual-train images ({CANDIDATE_POOL_FRACTION:.0%})')
print(f'TACS selected contexts per query: {CONTEXT_TOP_K}')
print(f'TACS uses full candidate pool during training: {TACS_USE_FULL_CANDIDATE_POOL}')
print(f'TACS Gumbel tau: {TACS_GUMBEL_TAU}, policy weight: {TACS_POLICY_WEIGHT}')
print(f'BBox crop enabled: {USE_BBOX_CROP}')
print(f'Train label range: {int(train_df.class_id.min())}..{int(train_df.class_id.max())}')
print(f'Test label range: {int(test_df.class_id.min())}..{int(test_df.class_id.max())}')
display(train_df.head())
display(val_df.head())
display(candidate_df.head())

## 2. Dataset v? model TACS

D?ng l?i ki?n tr?c TACS v? checkpoint ?? train. Candidate image bank ???c gi? tr?n CPU ?? d?nh VRAM cho DINOv3 v? SigLIP 2.


In [ ]:
class SimpleImageDataset(Dataset):
    def __init__(self, annotations, transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.crop_bbox = crop_bbox

    def __len__(self):
        return len(self.annotations)

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = min(image.width, max(left + 1, int(row['bbox_x2'])))
            bottom = min(image.height, max(top + 1, int(row['bbox_y2'])))
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        image = self.transform(self._open_image(row))
        return image, int(index), row['fname'], int(row['class_id'])

class StanfordCarsTACSQueryDataset(Dataset):
    def __init__(self, annotations, transform, crop_bbox=False):
        self.annotations = annotations.reset_index(drop=True).copy()
        self.transform = transform
        self.crop_bbox = crop_bbox

    def __len__(self):
        return len(self.annotations)

    def _open_image(self, row):
        image = Image.open(row['image_path']).convert('RGB')
        if self.crop_bbox:
            left = max(0, int(row['bbox_x1']) - 1)
            top = max(0, int(row['bbox_y1']) - 1)
            right = min(image.width, max(left + 1, int(row['bbox_x2'])))
            bottom = min(image.height, max(top + 1, int(row['bbox_y2'])))
            image = image.crop((left, top, right, bottom))
        return image

    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        return {
            'query_image': self.transform(self._open_image(row)),
            'label': torch.tensor(int(row['class_id']), dtype=torch.long),
            'fname': row['fname'],
            'row_id': torch.tensor(int(row['row_id']), dtype=torch.long),
        }

In [ ]:
import gc
import torch
import torch.nn.functional as F
from pathlib import Path

def resolve_dinov3_weights():
    path = Path(DINOV3_WEIGHTS)
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy file weight: {path}')
    return str(path)

def load_dinov3_model(repo_dir, model_name, weights_path):
    repo_dir = Path(repo_dir)
    if not repo_dir.is_dir():
        raise FileNotFoundError(f'Không tìm thấy thư mục mã nguồn: {repo_dir}')
    weights_path = Path(weights_path)
    if not weights_path.is_file():
        raise FileNotFoundError(f'Không tìm thấy file weight: {weights_path}')
    return torch.hub.load(str(repo_dir), model_name, source='local', weights=str(weights_path))

def load_dinov3_backbone():
    return load_dinov3_model(DINOV3_REPO_DIR, DINOV3_MODEL_NAME, resolve_dinov3_weights())

def ensure_dinov3_assets():
    resolve_dinov3_weights()
    repo_dir = Path(DINOV3_REPO_DIR)
    if not repo_dir.is_dir():
        raise FileNotFoundError(f'Không tìm thấy thư mục mã nguồn: {repo_dir}')
    print('DINOv3 assets đã sẵn sàng từ Kaggle Input.')

def extract_cls_features(backbone, images):
    features = backbone(images)
    if isinstance(features, dict):
        if 'x_norm_clstoken' in features:
            return features['x_norm_clstoken']
        if 'x_norm_cls_token' in features:
            return features['x_norm_cls_token']
        if 'x_norm_patchtokens' in features:
            return features['x_norm_patchtokens'].mean(dim=1)
        raise KeyError(f'Khong tim thay CLS token trong output keys: {list(features)}')
    return features[:, 0] if torch.is_tensor(features) and features.ndim == 3 else features

def extract_frozen_dino_embeddings(annotations, backbone, desc):
    dataset = SimpleImageDataset(annotations, eval_transform, crop_bbox=USE_BBOX_CROP)
    loader = DataLoader(dataset, batch_size=RETRIEVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True,persistent_workers=True if NUM_WORKERS > 0 else False)
    embeddings = []
    backbone.eval()
    for parameter in backbone.parameters():
        parameter.requires_grad_(False)
    with torch.inference_mode():
        for images, *_ in tqdm(loader, desc=desc, leave=False):
            images = images.to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                features = extract_cls_features(backbone, images)
            features = F.normalize(features.float(), dim=1)
            embeddings.append(features.cpu())
    return torch.cat(embeddings, dim=0)

class DinoV3FrozenSimilarityContextClassifier(nn.Module):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=384, allow_feature_fallback=False):
        super().__init__()
        self.backbone = load_dinov3_model(repo_dir, model_name, weights_path)
        self.feature_dim = feature_dim
        self.allow_feature_fallback = allow_feature_fallback
        self.active_fusion_mode = 'topk_token_attention'
        self.context_proj = nn.Linear(feature_dim, feature_dim)
        self.query_proj = nn.Linear(feature_dim, feature_dim)
        self.context_attention = nn.Linear(feature_dim, 1)
        self.fusion_norm = nn.LayerNorm(feature_dim)
        self.token_head = nn.Sequential(nn.Dropout(p=TOKEN_HEAD_DROPOUT), nn.Linear(feature_dim, num_classes))
        self.feature_fusion = nn.Sequential(nn.Linear(feature_dim * 2, feature_dim), nn.GELU(), nn.Dropout(p=TOKEN_HEAD_DROPOUT), nn.Linear(feature_dim, num_classes))
        for param in self.backbone.parameters():
            param.requires_grad_(False)

    def _prefix_token_count(self):
        if hasattr(self.backbone, 'num_prefix_tokens'):
            return int(self.backbone.num_prefix_tokens)
        storage_tokens = getattr(self.backbone, 'n_storage_tokens', None)
        if storage_tokens is None:
            storage_tokens = getattr(self.backbone, 'num_register_tokens', 0)
        return 1 + int(storage_tokens or 0)

    def _patch_grid(self, images):
        patch_size = getattr(self.backbone, 'patch_size', 16)
        if isinstance(patch_size, (tuple, list)):
            patch_h, patch_w = int(patch_size[0]), int(patch_size[1])
        else:
            patch_h = patch_w = int(patch_size)
        return images.shape[-2] // patch_h, images.shape[-1] // patch_w

    def _unwrap_tokens(self, prepared):
        if torch.is_tensor(prepared):
            return prepared
        if isinstance(prepared, (tuple, list)):
            for item in prepared:
                if torch.is_tensor(item):
                    return item
        if isinstance(prepared, dict):
            for key in ('x', 'tokens', 'last_hidden_state'):
                if key in prepared and torch.is_tensor(prepared[key]):
                    return prepared[key]
            for item in prepared.values():
                if torch.is_tensor(item):
                    return item
        raise TypeError(f'Cannot unwrap DINO token output of type {type(prepared)}')

    def _prepare_tokens_with_rope(self, images):
        try:
            prepared = self.backbone.prepare_tokens_with_masks(images, None)
        except TypeError:
            prepared = self.backbone.prepare_tokens_with_masks(images)
        tokens = self._unwrap_tokens(prepared)
        rope = None
        if hasattr(self.backbone, 'rope_embed'):
            grid_h, grid_w = self._patch_grid(images)
            try:
                rope = self.backbone.rope_embed(H=grid_h, W=grid_w)
            except TypeError:
                rope = self.backbone.rope_embed(grid_h, grid_w)
        return tokens, rope

    def _repeat_rope_for_query_context(self, rope):
        if rope is None or not torch.is_tensor(rope):
            return rope
        try:
            return rope.repeat_interleave(2, dim=-2)
        except RuntimeError:
            return rope

    def _run_blocks(self, tokens, rope):
        for block in self.backbone.blocks:
            if rope is None:
                tokens = block(tokens)
                continue
            try:
                tokens = block(tokens, rope=rope)
            except TypeError:
                try:
                    tokens = block(tokens, rope)
                except TypeError:
                    tokens = block(tokens)
        if hasattr(self.backbone, 'norm'):
            tokens = self.backbone.norm(tokens)
        return tokens

    def _forward_token_concat_features(self, query_images, context_images):
        query_tokens, query_rope = self._prepare_tokens_with_rope(query_images)
        context_tokens, _ = self._prepare_tokens_with_rope(context_images)
        prefix_tokens = self._prefix_token_count()
        pair_tokens = torch.cat([query_tokens[:, :prefix_tokens], query_tokens[:, prefix_tokens:], context_tokens[:, prefix_tokens:]], dim=1)
        pair_rope = self._repeat_rope_for_query_context(query_rope)
        pair_tokens = self._run_blocks(pair_tokens, pair_rope)
        return pair_tokens[:, 0]

    def _pool_backbone_features(self, images):
        features = self.backbone.forward_features(images)
        if isinstance(features, dict):
            if 'x_norm_clstoken' in features:
                return features['x_norm_clstoken']
            if 'x_norm_patchtokens' in features:
                return features['x_norm_patchtokens'].mean(dim=1)
            for item in features.values():
                if torch.is_tensor(item):
                    return item[:, 0] if item.ndim == 3 else item
        if torch.is_tensor(features):
            return features[:, 0] if features.ndim == 3 else features
        raise TypeError(f'Cannot pool DINO features of type {type(features)}')

    def _forward_feature_fusion(self, query_images, context_images):
        if context_images.ndim == 5:
            context_images = context_images[:, 0]
        query_features = self._pool_backbone_features(query_images)
        context_features = self._pool_backbone_features(context_images)
        self.active_fusion_mode = 'feature_fusion'
        return self.feature_fusion(torch.cat([query_features, context_features], dim=1))

    def _forward_topk_token_attention(self, query_images, context_images, context_scores=None, context_mask=None):
        if context_images.ndim == 4:
            context_images = context_images.unsqueeze(1)
        batch_size, num_contexts, channels, height, width = context_images.shape
        query_repeated = query_images.unsqueeze(1).expand(-1, num_contexts, -1, -1, -1).reshape(batch_size * num_contexts, channels, height, width)
        context_flat = context_images.reshape(batch_size * num_contexts, channels, height, width)
        pair_features = self._forward_token_concat_features(query_repeated, context_flat).reshape(batch_size, num_contexts, self.feature_dim)
        if context_mask is None:
            context_mask = torch.ones(batch_size, num_contexts, device=pair_features.device, dtype=torch.bool)
        else:
            context_mask = context_mask.to(device=pair_features.device, dtype=torch.bool)
            if context_mask.shape != (batch_size, num_contexts) or not context_mask.any(dim=1).all():
                raise ValueError('Each query must have at least one active context.')
        mask_values = context_mask.unsqueeze(-1).to(pair_features.dtype)
        query_summary = (pair_features * mask_values).sum(dim=1) / mask_values.sum(dim=1).clamp_min(1.0)
        attention_hidden = torch.tanh(self.context_proj(pair_features) + self.query_proj(query_summary).unsqueeze(1))
        attention_logits = self.context_attention(attention_hidden).squeeze(-1)
        if context_scores is not None:
            if context_scores.ndim == 1:
                context_scores = context_scores.unsqueeze(1)
            attention_logits = attention_logits + RETRIEVAL_SCORE_WEIGHT * context_scores.to(attention_logits.device, dtype=attention_logits.dtype)
        attention_logits = attention_logits.masked_fill(~context_mask, torch.finfo(attention_logits.dtype).min)
        attention_weights = torch.softmax(attention_logits, dim=1)
        fused_features = torch.sum(pair_features * attention_weights.unsqueeze(-1), dim=1)
        fused_features = self.fusion_norm(fused_features)
        self.active_fusion_mode = 'topk_token_attention'
        return fused_features, attention_weights

    def forward(self, query_images, context_images, context_scores=None, return_features=False, context_mask=None):
        try:
            fused_features, attention_weights = self._forward_topk_token_attention(
                query_images, context_images, context_scores=context_scores, context_mask=context_mask
            )
            logits = self.token_head(fused_features)
            if return_features:
                return logits, F.normalize(fused_features.float(), dim=1), attention_weights
            return logits
        except Exception as exc:
            if self.allow_feature_fallback:
                print(f'Top-k token attention failed, fallback to feature fusion: {type(exc).__name__}: {exc}')
                logits = self._forward_feature_fusion(query_images, context_images)
                if return_features:
                    return logits, F.normalize(logits.float(), dim=1), None
                return logits
            raise RuntimeError('Top-k token attention failed. Keep allow_feature_fallback=False for debugging.') from exc

class DinoV3GatedTopKContextClassifier(DinoV3FrozenSimilarityContextClassifier):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=384, allow_feature_fallback=False):
        super().__init__(
            repo_dir=repo_dir,
            model_name=model_name,
            weights_path=weights_path,
            num_classes=num_classes,
            feature_dim=feature_dim,
            allow_feature_fallback=allow_feature_fallback,
        )
        self.query_norm = nn.LayerNorm(feature_dim)
        self.context_norm = nn.LayerNorm(feature_dim)
        self.final_norm = nn.LayerNorm(feature_dim)

        self.query_head = nn.Sequential(
            nn.Dropout(p=TOKEN_HEAD_DROPOUT),
            nn.Linear(feature_dim, num_classes),
        )

        self.fusion_gate = nn.Sequential(
            nn.Linear(feature_dim * 4, feature_dim),
            nn.GELU(),
            nn.Dropout(p=QUERY_CONTEXT_GATE_DROPOUT),
            nn.Linear(feature_dim, 1),
        )
        nn.init.constant_(self.fusion_gate[-1].bias, QUERY_CONTEXT_GATE_BIAS)
        self.active_fusion_mode = "query_plus_gated_topk_context"

    def forward(self, query_images, context_images, context_scores=None, return_features=False, context_mask=None):
        try:
            context_features, attention_weights = self._forward_topk_token_attention(
                query_images,
                context_images,
                context_scores=context_scores,
                context_mask=context_mask,
            )
            query_features = self._pool_backbone_features(query_images)

            query_features = self.query_norm(query_features)
            context_features = self.context_norm(context_features)

            gate_input = torch.cat(
                [
                    query_features,
                    context_features,
                    query_features * context_features,
                    torch.abs(query_features - context_features),
                ],
                dim=1,
            )
            gate = torch.sigmoid(self.fusion_gate(gate_input))

            query_logits = self.query_head(query_features)
            context_logits = self.token_head(context_features)
            logits = query_logits + gate * context_logits

            fused_features = self.final_norm(query_features + gate * context_features)
            self.active_fusion_mode = "query_plus_gated_topk_context"

            if return_features:
                return logits, F.normalize(fused_features.float(), dim=1), attention_weights
            return logits

        except Exception as exc:
            if self.allow_feature_fallback:
                print(f"Gated fusion failed, fallback to parent model: {type(exc).__name__}: {exc}")
                return super().forward(query_images, context_images, context_scores=context_scores, return_features=return_features)
            raise

class TACSSelector(nn.Module):
    def __init__(self, temperature=1.0, normalize=True):
        super().__init__()
        self.temperature = float(temperature)
        self.normalize = bool(normalize)
        self.logit_scale = nn.Parameter(torch.tensor(1.0))

    def score(self, query_features, candidate_features):
        query = query_features.float()
        candidates = candidate_features.float()
        if self.normalize:
            query = F.normalize(query, dim=-1)
            candidates = F.normalize(candidates, dim=-1)
        scores = query @ candidates.T
        return scores * self.logit_scale.clamp(0.01, 100.0) / max(self.temperature, 1e-6)

class DinoV3TACSClassifier(DinoV3GatedTopKContextClassifier):
    def __init__(self, repo_dir, model_name, weights_path, num_classes, feature_dim=384, allow_feature_fallback=False):
        super().__init__(
            repo_dir=repo_dir,
            model_name=model_name,
            weights_path=weights_path,
            num_classes=num_classes,
            feature_dim=feature_dim,
            allow_feature_fallback=allow_feature_fallback,
        )
        self.selector = TACSSelector(
            temperature=TACS_SELECTOR_TEMPERATURE,
            normalize=True,
        )
        self.active_fusion_mode = 'tacs_full_pool_shared_backbone_selector'

    def selector_encode_images(self, images):
        return self._pool_backbone_features(images)

    def score_full_candidate_pool(self, query_images, candidate_feature_bank):
        query_features = self.selector_encode_images(query_images)
        selector_scores = self.selector.score(query_features, candidate_feature_bank.to(query_features.device))
        return selector_scores, query_features

    def select_full_pool(self, query_images, candidate_feature_bank, use_gumbel=True):
        selector_scores, query_features = self.score_full_candidate_pool(query_images, candidate_feature_bank)
        selection = 'gumbel' if self.training and use_gumbel else 'greedy'
        selected_indices, selection_weights, selected_mmr_scores, _ = select_diverse_indices(
            selector_scores, candidate_feature_bank, CONTEXT_TOP_K, selection=selection
        )
        return {
            'selector_scores': selector_scores,
            'selection_weights': selection_weights,
            'selected_indices': selected_indices,
            'selected_mmr_scores': selected_mmr_scores,
            'query_features': query_features,
            'selector_entropy': torch.distributions.Categorical(logits=selector_scores.float()).entropy().mean(),
        }

    def forward_selected_context(self, query_images, selected_context_images, return_features=False, context_mask=None):
        if selected_context_images.ndim == 4:
            selected_context_images = selected_context_images.unsqueeze(1)
        if selected_context_images.ndim != 5:
            raise ValueError('Contexts must have shape [B, K, C, H, W] or [B, C, H, W].')
        return super().forward(query_images, selected_context_images, context_scores=None,
                               return_features=return_features, context_mask=context_mask)

    def forward_query_only(self, query_images):
        query_features = self._pool_backbone_features(query_images)
        if hasattr(self, 'query_norm') and hasattr(self, 'query_head'):
            query_features = self.query_norm(query_features)
            return self.query_head(query_features)
        return self.token_head(query_features)

def build_candidate_image_bank(annotations, transform, device, dtype):
    dataset = SimpleImageDataset(annotations, transform, crop_bbox=USE_BBOX_CROP)
    loader = DataLoader(dataset, batch_size=RETRIEVAL_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    images_bank = []
    labels = []
    fnames = []
    for images, labels_batch, fnames_batch, *_ in tqdm(loader, desc='build candidate image bank', leave=False):
        images_bank.append(images.to(device=device, dtype=dtype, non_blocking=True).contiguous())
        labels.extend([int(x) for x in labels_batch.tolist()])
        fnames.extend(fnames_batch)
    return torch.cat(images_bank, dim=0), torch.tensor(labels, dtype=torch.long, device=device), fnames

def refresh_candidate_feature_bank(network, candidate_image_bank, batch_size=CANDIDATE_FEATURE_BANK_BATCH_SIZE):
    was_training = network.training
    network.eval()
    features = []
    with torch.inference_mode():
        for start in tqdm(range(0, len(candidate_image_bank), batch_size), desc='refresh full candidate feature bank', leave=False):
            end = min(start + batch_size, len(candidate_image_bank))
            images = candidate_image_bank[start:end].to(DEVICE, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=DEVICE.type == 'cuda'):
                feats = network.selector_encode_images(images)
            features.append(feats.float().detach().cpu())
    if was_training:
        network.train()
    return torch.cat(features, dim=0)

In [ ]:
import torch
import torch.nn.functional as F

def gather_bank_images(indices):
    return candidate_image_bank[indices.detach().to(candidate_image_bank.device)].to(DEVICE, non_blocking=True).float()

def adaptive_context_mask(selected_mmr_scores, threshold):
    gaps = selected_mmr_scores[:, :1] - selected_mmr_scores
    keep = gaps <= float(threshold)
    keep[:, 0] = True
    for rank in range(1, keep.size(1)):
        keep[:, rank] &= keep[:, rank - 1]
    return keep

def select_diverse_indices(selector_scores, candidate_feature_bank, max_k, selection='greedy'):
    scores = selector_scores.float()
    relevance = (scores - scores.mean(1, keepdim=True)) / scores.std(
        1, keepdim=True, unbiased=False
    ).clamp_min(1e-6)
    features = F.normalize(candidate_feature_bank.to(scores.device).float(), dim=1)
    max_k = min(int(max_k), scores.size(1))
    chosen = torch.zeros_like(scores, dtype=torch.bool)
    redundancy = torch.zeros_like(scores)
    indices, weights, mmr_scores, log_probs = [], [], [], []
    for _ in range(max_k):
        mmr = (relevance - DIVERSITY_WEIGHT * redundancy).masked_fill(
            chosen, torch.finfo(relevance.dtype).min
        )
        if selection == 'gumbel':
            weight = F.gumbel_softmax(mmr, tau=TACS_GUMBEL_TAU, hard=True, dim=1)
            index = weight.argmax(1)
        elif selection == 'sample':
            dist = torch.distributions.Categorical(logits=mmr)
            index = dist.sample()
            log_probs.append(dist.log_prob(index))
            weight = F.one_hot(index, scores.size(1)).to(scores.dtype)
        else:
            index = mmr.argmax(1)
            weight = F.one_hot(index, scores.size(1)).to(scores.dtype)
        indices.append(index)
        weights.append(weight)
        mmr_scores.append(mmr.gather(1, index[:, None]).squeeze(1))
        chosen = chosen.scatter(1, index[:, None], True)
        similarity = features @ features[index].T
        redundancy = torch.maximum(redundancy, similarity.T.clamp_min(0.0))
    log_probs = torch.stack(log_probs, dim=1) if log_probs else None
    return torch.stack(indices, 1), torch.stack(weights, 1), torch.stack(mmr_scores, 1), log_probs

# Threshold 1.5 was selected on validation in the saved training notebook.
# Change only if using another checkpoint/run.
TACS_ADAPTIVE_THRESHOLD = 1.5


In [ ]:

# EDIT THESE TWO PATHS to point to the Kaggle Input datasets you attach.
TACS_CHECKPOINT_PATH = Path(
    "/kaggle/input/datasets/YOUR_USERNAME/tacs-adaptive-sun/best_val_checkpoint.pt"
)
SIGLIP2_MODEL_DIR = Path(
    "/kaggle/input/datasets/YOUR_USERNAME/siglip2-base-patch16-224"
)

if not TACS_CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"TACS checkpoint not found: {TACS_CHECKPOINT_PATH}. "
        "Upload best_val_checkpoint.pt from the trained Kaggle notebook as a Kaggle Dataset."
    )
if not SIGLIP2_MODEL_DIR.is_dir() or not (SIGLIP2_MODEL_DIR / "config.json").is_file():
    raise FileNotFoundError(
        f"SigLIP 2 folder not found: {SIGLIP2_MODEL_DIR}. "
        "Upload the complete google/siglip2-base-patch16-224 folder as a Kaggle Dataset."
    )

ensure_dinov3_assets()
assert len(class_names) == NUM_CLASSES == 397
assert not TRAIN_ON_FULL_TRAIN, "Fusion calibration requires a true held-out validation split."

# Keep the pixel bank in CPU RAM to leave GPU memory for both encoders.
candidate_image_bank, candidate_labels_tensor, candidate_fnames = build_candidate_image_bank(
    candidate_df,
    eval_transform,
    device=torch.device("cpu"),
    dtype=torch.float16,
)
print("Candidate image bank:", tuple(candidate_image_bank.shape), candidate_image_bank.device)

model = DinoV3TACSClassifier(
    DINOV3_REPO_DIR,
    DINOV3_MODEL_NAME,
    resolve_dinov3_weights(),
    NUM_CLASSES,
    FEATURE_DIM,
    allow_feature_fallback=False,
).to(DEVICE)

checkpoint = torch.load(TACS_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state = checkpoint.get("model_state_dict")
if not isinstance(state, dict):
    raise KeyError("Checkpoint does not contain model_state_dict.")
model.load_state_dict(state, strict=True)
model.eval()

candidate_feature_bank = refresh_candidate_feature_bank(model, candidate_image_bank)
print("Loaded TACS checkpoint:", TACS_CHECKPOINT_PATH)
print("Candidate feature bank:", tuple(candidate_feature_bank.shape))
print("TACS validation-selected adaptive threshold:", TACS_ADAPTIVE_THRESHOLD)


## 3. N?p SigLIP 2 offline

SigLIP 2 ???c n?p t? th? m?c Kaggle Input v?i local_files_only=True. Text embeddings c?a c?c t?n l?p ???c t?nh m?t l?n, d?ng prompt ensemble ba m?u.


In [ ]:

import transformers
from packaging.version import Version
from transformers import AutoConfig, AutoModel, AutoProcessor

if Version(transformers.__version__) < Version("4.50.0"):
    raise RuntimeError(
        f"SigLIP 2 needs a recent Transformers build; found {transformers.__version__}. "
        "Because Kaggle Internet is disabled, attach an offline wheel or use an environment with Transformers >= 4.50."
    )

siglip_dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
siglip_processor = AutoProcessor.from_pretrained(
    str(SIGLIP2_MODEL_DIR), local_files_only=True
)
siglip_config = AutoConfig.from_pretrained(
    str(SIGLIP2_MODEL_DIR), local_files_only=True
)
text_config = siglip_config.text_config
tokenizer = siglip_processor.tokenizer
vocab_size = int(text_config.vocab_size)

# Some SigLIP2 configs retain CLIP BOS/EOS defaults outside the 32k vocabulary.
# These IDs are not used to encode prompts; align config metadata with the tokenizer.
for token_attr in ("bos_token_id", "eos_token_id"):
    token_id = getattr(tokenizer, token_attr, None)
    if isinstance(token_id, (list, tuple)):
        token_id = token_id[0] if token_id else None
    if not isinstance(token_id, int) or not 0 <= token_id < vocab_size:
        token_id = None
    setattr(text_config, token_attr, token_id)

siglip_model = AutoModel.from_pretrained(
    str(SIGLIP2_MODEL_DIR),
    config=siglip_config,
    local_files_only=True,
    dtype=siglip_dtype,
).to(DEVICE).eval()
for parameter in siglip_model.parameters():
    parameter.requires_grad_(False)

def siglip_output_features(output, name):
    if torch.is_tensor(output):
        return output
    pooled = getattr(output, "pooler_output", None)
    if torch.is_tensor(pooled):
        return pooled
    if isinstance(output, dict):
        for key in ("pooler_output", "text_embeds", "image_embeds", "embeds"):
            value = output.get(key)
            if torch.is_tensor(value):
                return value
    if isinstance(output, (tuple, list)):
        for value in output:
            if torch.is_tensor(value) and value.ndim == 2:
                return value
    raise TypeError(f"Cannot extract pooled {name} features from {type(output).__name__}")

def sun_class_prompt_name(class_name):
    parts = [part for part in str(class_name).replace("\\", "/").strip("/").split("/") if part]
    if len(parts) > 1 and len(parts[0]) == 1:
        parts = parts[1:]
    label = " ".join(parts).replace("_", " ").replace("-", " ").lower()
    return " ".join(label.split())

class_labels = [sun_class_prompt_name(name) for name in class_names]
assert len(class_labels) == 397 and all(class_labels)

PROMPT_TEMPLATES = [
    "this is a photo of {}.",
    "a photo of the {} scene.",
    "a scene showing {}.",
]
prompts = [template.format(label) for label in class_labels for template in PROMPT_TEMPLATES]

text_feature_chunks = []
with torch.inference_mode():
    for start in tqdm(range(0, len(prompts), 128), desc="encode SUN397 text prompts"):
        token_inputs = siglip_processor(
            text=prompts[start:start + 128],
            padding="max_length",
            max_length=64,
            return_tensors="pt",
        )
        token_inputs = {key: value.to(DEVICE) for key, value in token_inputs.items() if torch.is_tensor(value)}
        if int(token_inputs["input_ids"].max()) >= vocab_size:
            raise ValueError(
                f"Tokenizer emits token ID {int(token_inputs['input_ids'].max())}, "
                f"but SigLIP2 text vocabulary has size {vocab_size}. "
                "The processor/tokenizer and model files are from different models."
            )
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            text_output = siglip_model.get_text_features(**token_inputs)
        text_features = siglip_output_features(text_output, "text")
        text_feature_chunks.append(F.normalize(text_features.float(), dim=1).cpu())

all_prompt_features = torch.cat(text_feature_chunks, dim=0).reshape(
    len(class_names), len(PROMPT_TEMPLATES), -1
)
siglip_text_features = F.normalize(all_prompt_features.mean(dim=1), dim=1).to(DEVICE)
SIGLIP_LOGIT_SCALE = (
    siglip_model.logit_scale.detach().float().exp()
    if hasattr(siglip_model, "logit_scale")
    else torch.tensor(1.0, device=DEVICE)
)
SIGLIP_LOGIT_BIAS = (
    siglip_model.logit_bias.detach().float()
    if hasattr(siglip_model, "logit_bias")
    else torch.tensor(0.0, device=DEVICE)
)
print("SigLIP 2:", SIGLIP2_MODEL_DIR)
print("Text embeddings:", tuple(siglip_text_features.shape))
print("Prompt examples:", prompts[:3])


## 4. Validation: zero-shot v? ch?n fusion

Ch?y TACS v? SigLIP 2 tr?n validation. Ch?n nhi?t ?? hai nh?nh v? h? s? fusion b?ng NLL validation; kh?ng d?ng test ?? ch?n tham s?.


In [ ]:

from torch.utils.data import Dataset, DataLoader
from PIL import Image

EVAL_BATCH_SIZE = 8
EVAL_NUM_WORKERS = 0

class SUNRowsWithPIL(Dataset):
    def __init__(self, annotations):
        self.annotations = annotations.reset_index(drop=True).copy()
    def __len__(self):
        return len(self.annotations)
    def __getitem__(self, index):
        row = self.annotations.iloc[index]
        with Image.open(row["image_path"]) as image:
            image = image.convert("RGB")
            image.load()
        return image, int(row["class_id"]), str(row["fname"])

def collate_sun_pil(batch):
    images, labels, fnames = zip(*batch)
    return list(images), torch.tensor(labels, dtype=torch.long), list(fnames)

@torch.inference_mode()
def infer_tacs_and_siglip(annotations, split_name):
    loader = DataLoader(
        SUNRowsWithPIL(annotations),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=EVAL_NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        collate_fn=collate_sun_pil,
    )
    tacs_logits_all, siglip_logits_all, labels_all, fnames_all = [], [], [], []
    model.eval()
    for pil_images, labels, fnames in tqdm(loader, desc=f"{split_name} TACS + SigLIP2"):
        labels_device = labels.to(DEVICE, non_blocking=True)
        query_images = torch.stack([eval_transform(image) for image in pil_images]).to(
            DEVICE, non_blocking=True
        )
        with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
            selection = model.select_full_pool(
                query_images, candidate_feature_bank.to(DEVICE), use_gumbel=False
            )
            context_mask = adaptive_context_mask(
                selection["selected_mmr_scores"], TACS_ADAPTIVE_THRESHOLD
            )
            selected_context_images = gather_bank_images(selection["selected_indices"])
            tacs_logits = model.forward_selected_context(
                query_images,
                selected_context_images,
                context_mask=context_mask,
                return_features=False,
            )
            image_inputs = siglip_processor(images=pil_images, return_tensors="pt")
            image_inputs = {
                key: value.to(DEVICE, non_blocking=True)
                for key, value in image_inputs.items()
                if torch.is_tensor(value)
            }
            image_output = siglip_model.get_image_features(**image_inputs)
            image_features = siglip_output_features(image_output, "image")
            image_features = F.normalize(image_features.float(), dim=1)
            siglip_logits = (
                SIGLIP_LOGIT_SCALE * (image_features @ siglip_text_features.T)
                + SIGLIP_LOGIT_BIAS
            )

        tacs_logits_all.append(tacs_logits.float().cpu())
        siglip_logits_all.append(siglip_logits.float().cpu())
        labels_all.append(labels_device.cpu())
        fnames_all.extend(fnames)

    return {
        "tacs_logits": torch.cat(tacs_logits_all),
        "siglip_logits": torch.cat(siglip_logits_all),
        "labels": torch.cat(labels_all),
        "fnames": fnames_all,
    }


In [ ]:

import numpy as np
from sklearn.metrics import f1_score

def classification_metrics(logits, labels):
    logits = torch.as_tensor(logits).float()
    labels = torch.as_tensor(labels).long()
    predictions = logits.argmax(dim=1)
    return {
        "accuracy": float((predictions == labels).float().mean()),
        "macro_f1": float(f1_score(labels.numpy(), predictions.numpy(), average="macro", zero_division=0)),
        "nll": float(F.cross_entropy(logits, labels)),
    }

val_outputs = infer_tacs_and_siglip(val_df, "validation")
val_tacs_logits = val_outputs["tacs_logits"]
val_siglip_logits = val_outputs["siglip_logits"]
val_labels = val_outputs["labels"]

TEMPERATURE_GRID = [0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
FUSION_ALPHA_GRID = np.linspace(0.0, 1.0, 21)
validation_grid = []
best = None
for t_tacs in TEMPERATURE_GRID:
    lp_tacs = F.log_softmax(val_tacs_logits / t_tacs, dim=1)
    for t_siglip in TEMPERATURE_GRID:
        lp_siglip = F.log_softmax(val_siglip_logits / t_siglip, dim=1)
        for alpha in FUSION_ALPHA_GRID:
            fused_log_probs = alpha * lp_tacs + (1.0 - alpha) * lp_siglip
            nll = float(F.nll_loss(fused_log_probs, val_labels))
            acc = float((fused_log_probs.argmax(1) == val_labels).float().mean())
            row = {
                "alpha_tacs": float(alpha),
                "temperature_tacs": float(t_tacs),
                "temperature_siglip": float(t_siglip),
                "nll": nll,
                "accuracy": acc,
            }
            validation_grid.append(row)
            key = (nll, -acc)
            if best is None or key < best[0]:
                best = (key, row)

best_fusion = best[1]
BEST_ALPHA_TACS = best_fusion["alpha_tacs"]
BEST_TEMP_TACS = best_fusion["temperature_tacs"]
BEST_TEMP_SIGLIP = best_fusion["temperature_siglip"]
validation_grid_df = pd.DataFrame(validation_grid).sort_values(
    ["nll", "accuracy"], ascending=[True, False], kind="stable"
)
validation_grid_df.to_csv(OUT_PUT_DIR / "siglip2_fusion_validation_grid.csv", index=False)
val_fused_log_probs = (
    BEST_ALPHA_TACS * F.log_softmax(val_tacs_logits / BEST_TEMP_TACS, dim=1)
    + (1.0 - BEST_ALPHA_TACS) * F.log_softmax(val_siglip_logits / BEST_TEMP_SIGLIP, dim=1)
)
val_metrics = pd.DataFrame([
    {"model": "TACS adaptive", **classification_metrics(val_tacs_logits, val_labels)},
    {"model": "SigLIP2 zero-shot", **classification_metrics(val_siglip_logits, val_labels)},
    {"model": "TACS + SigLIP2 fusion", **classification_metrics(val_fused_log_probs, val_labels)},
])
print("Fusion parameters selected on validation:", best_fusion)
display(val_metrics)
display(validation_grid_df.head(10))


## 5. Official test

Sau khi kh?a h? s? tr?n validation, ch?y test m?t l?n v? l?u metrics/predictions. So s?nh TACS, SigLIP 2 zero-shot v? late-fusion.


In [ ]:

test_outputs = infer_tacs_and_siglip(test_df, "official test")
test_tacs_logits = test_outputs["tacs_logits"]
test_siglip_logits = test_outputs["siglip_logits"]
test_labels = test_outputs["labels"]

test_fused_log_probs = (
    BEST_ALPHA_TACS * F.log_softmax(test_tacs_logits / BEST_TEMP_TACS, dim=1)
    + (1.0 - BEST_ALPHA_TACS) * F.log_softmax(test_siglip_logits / BEST_TEMP_SIGLIP, dim=1)
)
test_metrics = pd.DataFrame([
    {"model": "TACS adaptive", **classification_metrics(test_tacs_logits, test_labels)},
    {"model": "SigLIP2 zero-shot", **classification_metrics(test_siglip_logits, test_labels)},
    {"model": "TACS + SigLIP2 fusion", **classification_metrics(test_fused_log_probs, test_labels)},
])
test_metrics["fusion_alpha_tacs"] = np.nan
test_metrics.loc[test_metrics["model"] == "TACS + SigLIP2 fusion", "fusion_alpha_tacs"] = BEST_ALPHA_TACS
test_metrics["temperature_tacs"] = np.nan
test_metrics.loc[test_metrics["model"] == "TACS + SigLIP2 fusion", "temperature_tacs"] = BEST_TEMP_TACS
test_metrics["temperature_siglip"] = np.nan
test_metrics.loc[test_metrics["model"] == "TACS + SigLIP2 fusion", "temperature_siglip"] = BEST_TEMP_SIGLIP
test_metrics.to_csv(OUT_PUT_DIR / "siglip2_tacs_test_metrics.csv", index=False)

predictions = pd.DataFrame({
    "fname": test_outputs["fnames"],
    "true_class_id": test_labels.numpy(),
    "true_class_name": [class_names[int(i)] for i in test_labels.tolist()],
    "tacs_pred_id": test_tacs_logits.argmax(1).numpy(),
    "siglip2_pred_id": test_siglip_logits.argmax(1).numpy(),
    "fusion_pred_id": test_fused_log_probs.argmax(1).numpy(),
})
for prefix, col in [("tacs", "tacs_pred_id"), ("siglip2", "siglip2_pred_id"), ("fusion", "fusion_pred_id")]:
    predictions[f"{prefix}_pred_name"] = [class_names[int(i)] for i in predictions[col]]
    predictions[f"{prefix}_correct"] = predictions[col] == predictions["true_class_id"]
predictions.to_csv(OUT_PUT_DIR / "siglip2_tacs_test_predictions.csv", index=False)

print("Official test metrics:")
display(test_metrics)
print("Fusion parameters (locked from validation):", best_fusion)
print("Saved to:", OUT_PUT_DIR.resolve())
display(predictions.head())
